# GLiNER2: Complete Feature Tour
### Every capability of the unified zero-shot information extraction model

---

## What is GLiNER2?

GLiNER2 is a **single 205M–340M parameter model** that unifies four NLP tasks that previously required separate models:

| Task | What it does | Old approach |
|------|-------------|-------------|
| **Named Entity Recognition (NER)** | Extract people, places, products, dates from text | Fine-tuned BERT/SpaCy per domain |
| **Text Classification** | Label text as positive/negative, topic, category | Fine-tuned classifier per label set |
| **Structured Extraction** | Pull out structured JSON from unstructured text | GPT-4 with JSON mode (expensive) |
| **Multi-Task** | All three above in ONE model call | Three separate model calls |

**Zero-shot** means you describe what you want in plain English — no training data or fine-tuning needed.

```
┌─────────────────────────────────────────────────────┐
│              GLiNER2 Model (~300MB)                 │
│                                                     │
│  Text Input + Schema ──► Single Forward Pass ──►   │
│                                                     │
│  ├── entities   { person: [...], company: [...] }  │
│  ├── domain     "machine_learning"                 │
│  ├── sentiment  "positive"                         │
│  └── product    [{ name: ..., price: ... }]        │
└─────────────────────────────────────────────────────┘
```

**Runs on CPU. No GPU required. Apache 2.0 license.**

---

## Table of Contents

| # | Feature | Real-world example used |
|---|---------|------------------------|
| 1 | [Setup & Model Loading](#s01) | — |
| 2 | [NER: Basic Entity Extraction](#s02) | Tech news article |
| 3 | [NER: With Entity Descriptions](#s03) | Medical case notes |
| 4 | [Classification: Single-label](#s04) | Customer support tickets |
| 5 | [Classification: Multi-label](#s05) | Product reviews |
| 6 | [Classification: With Confidence Scores](#s06) | News headlines |
| 7 | [Structured Extraction: Basic](#s07) | Job postings |
| 8 | [Structured Extraction: choices/enum fields](#s08) | Restaurant reservations |
| 9 | [Structured Extraction: Per-field Thresholds](#s09) | Financial data |
| 10 | [RegexValidator: Post-processing Filters](#s10) | Contact information |
| 11 | [Multi-task: Everything in One Call](#s11) | Earnings call transcript |
| 12 | [Batch Processing](#s12) | E-commerce product catalogue |
| 13 | [Schema Builder API: Full Control](#s13) | HR hiring pipeline |
| 14 | [Schema Caching & Reuse](#s14) | Document processing pipeline |
| 15 | [base vs large: Accuracy vs Speed](#s15) | Benchmark comparison |
| 16 | [Real-world Pipeline: News Intelligence](#s16) | News analysis end-to-end |

---
## Section 1: Setup & Model Loading <a id='s01'></a>

> **⚠️ First run:** GLiNER2 downloads model weights (~300–500 MB) from HuggingFace on first load.
> This takes 1–5 minutes. Subsequent runs use the local cache.

### Two models available:
| Model | Parameters | Best for |
|-------|-----------|----------|
| `fastino/gliner2-base-v1` | 205M | Fast prototyping, high-throughput production |
| `fastino/gliner2-large-v1` | 340M | Higher accuracy, complex schemas, production quality |

In [1]:
from gliner2 import GLiNER2, RegexValidator
import re
import json
import time

def pp(result, indent=2):
    """Pretty-print extraction results."""
    print(json.dumps(result, indent=indent, default=str))

# Load base model (faster, good for demos)
print("Loading GLiNER2 base model...")
t0 = time.perf_counter()
model = GLiNER2.from_pretrained("fastino/gliner2-base-v1")
print(f"Model loaded in {time.perf_counter() - t0:.1f}s")
print(f"Model type : {type(model).__name__}")
print("Ready.")

/Users/sourangshupal/Downloads/metadata-hybrid-rag/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading GLiNER2 base model...
🧠 Model Configuration
Encoder model      : microsoft/deberta-v3-base
Counting layer     : count_lstm_v2
Token pooling      : first
Model loaded in 51.4s
Model type : GLiNER2
Ready.


---
## Section 2: NER — Basic Entity Extraction <a id='s02'></a>

The most fundamental use case. You give GLiNER2 a list of **entity type names** and it finds all mentions in the text — **zero training data needed**.

**Method:** `model.extract_entities(text, entity_types)`

You can pass entity types as:
- A **list of strings**: `["person", "company", "location"]`
- A **dict**: `{"person": "description", ...}` (see Section 3 for why this is better)

Let's use a tech news article.

In [2]:
# --- Sample: Tech news article ---
tech_news = """
NVIDIA CEO Jensen Huang announced at CES 2025 in Las Vegas that the company's new
Blackwell Ultra GPU architecture will ship in Q2 2025. The RTX 5090, priced at $1,999,
delivers 3x the performance of the RTX 4090. Microsoft and Google have already placed
large orders, while AMD's Lisa Su confirmed their competing MI400 chip targets the
same datacenter market. The announcement sent NVIDIA's stock up 8% on the NYSE.
"""

# Simple list of entity types
result = model.extract_entities(
    tech_news,
    ["person", "company", "product", "location", "date", "price"]
)

print("=== Extracted Entities ===")
pp(result)

=== Extracted Entities ===
{
  "entities": {
    "person": [
      "Jensen Huang",
      "Lisa Su"
    ],
    "company": [
      "Google",
      "NVIDIA",
      "AMD",
      "Microsoft",
      "NYSE"
    ],
    "product": [
      "RTX 5090",
      "MI400",
      "RTX 4090",
      "Blackwell Ultra GPU"
    ],
    "location": [
      "Las Vegas"
    ],
    "date": [
      "Q2 2025"
    ],
    "price": [
      "$1,999"
    ]
  }
}


In [3]:
# Try a completely different domain — same model, no retraining!
# Sample: Sports news
sports_news = """
Real Madrid defeated Manchester City 3-1 at the Bernabeu Stadium on Tuesday.
Kylian Mbappe scored twice in the first half, with Vinicius Jr adding a third
in the 78th minute. City's only goal came from Erling Haaland in stoppage time.
Manager Carlo Ancelotti called it "a perfect performance" ahead of next week's
LaLiga match against Barcelona at Camp Nou.
"""

result = model.extract_entities(
    sports_news,
    ["player", "team", "stadium", "manager", "competition", "score"]
)

print("=== Sports NER (same model!) ===")
pp(result)
print()
print("Key point: You defined domain-specific entity types (player, team, stadium)")
print("that don't exist in any pre-trained NER model — GLiNER2 understands them zero-shot.")

=== Sports NER (same model!) ===
{
  "entities": {
    "player": [
      "Kylian Mbappe",
      "Erling Haaland",
      "Vinicius Jr"
    ],
    "team": [
      "Manchester City",
      "Real Madrid",
      "Barcelona"
    ],
    "stadium": [
      "Camp Nou",
      "Bernabeu Stadium"
    ],
    "manager": [
      "Carlo Ancelotti"
    ],
    "competition": [
      "LaLiga"
    ],
    "score": [
      "3-1"
    ]
  }
}

Key point: You defined domain-specific entity types (player, team, stadium)
that don't exist in any pre-trained NER model — GLiNER2 understands them zero-shot.


---
## Section 3: NER with Entity Descriptions <a id='s03'></a>

Passing a **description** for each entity type dramatically improves accuracy, especially for:
- Ambiguous terms ("Apple" — company or fruit?)
- Domain-specific concepts with technical meaning
- Distinguishing overlapping entity types

**Rule of thumb:** Always use descriptions for production use. They are free — they cost no extra computation.

In [4]:
# Sample: Medical case notes
# This is where entity description is critical — medical terms are highly ambiguous
medical_text = """
Patient John M., 67-year-old male, presented to Dr. Sarah Chen at Memorial Hospital
on 14-Jan-2025 with acute chest pain. BP was 145/90 mmHg, HR 92 bpm.
History includes Type 2 diabetes (metformin 500mg twice daily) and hypertension
(lisinopril 10mg). ECG showed ST-elevation. Troponin I at 2.3 ng/mL (normal < 0.04).
Diagnosed with STEMI. Patient admitted to ICU, started on heparin and aspirin 325mg.
PCI scheduled for 08:00 on 15-Jan-2025. No known drug allergies.
"""

# WITHOUT descriptions — model guesses intent
print("--- WITHOUT descriptions ---")
result_basic = model.extract_entities(
    medical_text,
    ["medication", "condition", "measurement", "person", "date"]
)
pp(result_basic)

print()
print("--- WITH descriptions — much more precise ---")
result_described = model.extract_entities(
    medical_text,
    {
        "medication":  "Drug names, medications, or pharmaceutical substances with dosage (e.g., metformin 500mg)",
        "condition":   "Medical conditions, diagnoses, diseases, or clinical findings (e.g., STEMI, hypertension)",
        "measurement": "Clinical measurements with units: blood pressure, heart rate, lab values (e.g., 145/90 mmHg)",
        "person":      "Patient names or doctor names, NOT hospital names",
        "facility":    "Hospital names, ICU, departments or clinical units",
        "date":        "Specific dates or times of clinical events",
        "procedure":   "Medical procedures, tests, or interventions (e.g., ECG, PCI, blood test)",
    }
)
pp(result_described)

--- WITHOUT descriptions ---
{
  "entities": {
    "medication": [
      "aspirin",
      "heparin",
      "metformin",
      "lisinopril"
    ],
    "condition": [
      "hypertension",
      "Type 2 diabetes",
      "STEMI",
      "ST-elevation",
      "drug allergies",
      "ECG",
      "PCI"
    ],
    "measurement": [
      "BP",
      "Troponin I",
      "HR",
      "ECG"
    ],
    "person": [
      "John M.",
      "Dr. Sarah Chen"
    ],
    "date": [
      "15-Jan-2025",
      "14-Jan-2025"
    ]
  }
}

--- WITH descriptions — much more precise ---
{
  "entities": {
    "medication": [
      "metformin 500mg",
      "heparin",
      "aspirin",
      "lisinopril"
    ],
    "condition": [
      "STEMI",
      "hypertension",
      "Type 2 diabetes",
      "ST-elevation",
      "drug allergies"
    ],
    "measurement": [
      "145/90 mmHg",
      "HR 92 bpm",
      "BP",
      "2.3 ng/mL"
    ],
    "person": [
      "John M.",
      "Dr. Sarah Chen"
    ],
    "facility": [

In [5]:
# Another example: Legal document — entity types don't exist in any standard NER model
legal_text = """
This Software License Agreement (the "Agreement") is entered into as of January 1, 2025,
between DataCorp Inc., a Delaware corporation ("Licensor") with offices at 123 Tech Drive,
San Francisco, CA, and StartupXYZ Ltd. ("Licensee"). The total license fee is $250,000 per year.
Breach of Section 4.2 triggers a penalty of $50,000. The Agreement is governed by
California law and any disputes go to arbitration in Santa Clara County.
Signed by: Alice Wang (CEO, DataCorp) and Bob Martinez (CTO, StartupXYZ).
"""

result = model.extract_entities(
    legal_text,
    {
        "party":        "Named parties to the contract — companies or individuals signing the agreement",
        "role":         "Job titles or legal roles (CEO, Licensor, Licensee)",
        "obligation":   "Specific duties, conditions, or contractual requirements",
        "financial_term": "Monetary amounts, fees, penalties, or financial obligations",
        "date":         "Dates, deadlines, or time-based conditions",
        "jurisdiction": "Governing law, legal jurisdiction, or venue for disputes",
    }
)

print("=== Legal Document Entity Extraction ===")
pp(result)

=== Legal Document Entity Extraction ===
{
  "entities": {
    "party": [
      "DataCorp Inc.",
      "StartupXYZ Ltd.",
      "Alice Wang",
      "Bob Martinez"
    ],
    "role": [
      "CEO",
      "CTO",
      "Licensor",
      "Licensee"
    ],
    "obligation": [
      "Breach of Section 4.2"
    ],
    "financial_term": [
      "$50,000",
      "$250,000"
    ],
    "date": [
      "January 1, 2025"
    ],
    "jurisdiction": [
      "California",
      "Delaware",
      "Santa Clara County"
    ]
  }
}


---
## Section 4: Text Classification — Single-label <a id='s04'></a>

GLiNER2 can classify text into any categories you define — no fine-tuning, just label names (and optionally descriptions).

**Method:** `model.classify_text(text, tasks_dict)`

The `tasks_dict` maps task names to lists of labels. You can run **multiple classification tasks simultaneously** in one call.

In [6]:
# Sample: Customer support tickets
tickets = [
    "My payment keeps failing with error code 402. I've tried 3 different cards and it still won't work. This is urgent, I need to complete this purchase today.",
    "Hi, I just wanted to say that the new dashboard update is absolutely beautiful! The charts load so much faster now. Really loving the product.",
    "How do I export my data to CSV? I've been looking through the docs but can't find the option anywhere. I need this for a report due tomorrow.",
    "Your competitor just launched the same feature for 40% less. I'm seriously considering switching unless you can match their pricing.",
]

print("=== Customer Support Ticket Classification ===")
print(f"{'Ticket':<20} {'Category':<18} {'Priority':<12} {'Sentiment'}")
print("-" * 70)

for i, ticket in enumerate(tickets, 1):
    result = model.classify_text(
        ticket,
        {
            "category":  ["billing", "technical_bug", "feature_request", "how_to_question", "compliment", "churn_risk"],
            "priority":  ["urgent", "high", "medium", "low"],
            "sentiment": ["positive", "negative", "neutral", "mixed"],
        }
    )
    preview = ticket[:35] + "..."
    print(f"  Ticket {i}: {preview}")
    print(f"    category={result['category']}, priority={result['priority']}, sentiment={result['sentiment']}")
    print()

=== Customer Support Ticket Classification ===
Ticket               Category           Priority     Sentiment
----------------------------------------------------------------------
  Ticket 1: My payment keeps failing with error...
    category=technical_bug, priority=urgent, sentiment=negative

  Ticket 2: Hi, I just wanted to say that the n...
    category=compliment, priority=high, sentiment=positive

  Ticket 3: How do I export my data to CSV? I'v...
    category=how_to_question, priority=high, sentiment=neutral

  Ticket 4: Your competitor just launched the s...
    category=billing, priority=high, sentiment=negative



In [7]:
# More example: News article topic classification
news_articles = [
    "The Federal Reserve raised interest rates by 25 basis points today, citing persistent inflation concerns despite recent cooling in CPI data.",
    "Scientists at MIT have developed a new CRISPR technique that can edit multiple genes simultaneously, potentially curing hereditary diseases.",
    "The new iPhone 17 features a foldable display and an upgraded neural engine that Apple claims delivers 40% better AI performance.",
    "Brazil defeated France 2-1 in the World Cup semifinals, setting up a final against Germany on Sunday at the Maracana Stadium.",
    "A 6.8 magnitude earthquake struck near Tokyo, triggering tsunami warnings along the Pacific coast. Authorities have issued evacuation orders.",
]

print("=== News Article Classification ===")
for i, article in enumerate(news_articles, 1):
    result = model.classify_text(
        article,
        {
            "topic":    ["finance", "science", "technology", "sports", "politics", "disaster", "health"],
            "urgency":  ["breaking", "important", "routine"],
            "geography": ["global", "us_focused", "europe", "asia", "latin_america"],
        }
    )
    print(f"  Article {i}: \"{article[:55]}...\"")
    print(f"    topic={result['topic']}, urgency={result['urgency']}, geography={result['geography']}")

=== News Article Classification ===
  Article 1: "The Federal Reserve raised interest rates by 25 basis p..."
    topic=finance, urgency=routine, geography=us_focused
  Article 2: "Scientists at MIT have developed a new CRISPR technique..."
    topic=science, urgency=important, geography=global
  Article 3: "The new iPhone 17 features a foldable display and an up..."
    topic=technology, urgency=routine, geography=global
  Article 4: "Brazil defeated France 2-1 in the World Cup semifinals,..."
    topic=sports, urgency=routine, geography=latin_america
  Article 5: "A 6.8 magnitude earthquake struck near Tokyo, triggerin..."
    topic=disaster, urgency=breaking, geography=asia


---
## Section 5: Text Classification — Multi-label <a id='s05'></a>

**Single-label** picks ONE best label. **Multi-label** picks ALL labels that exceed the `cls_threshold`.

Use multi-label when:
- Content naturally spans multiple categories (e.g., a Python + Machine Learning tutorial)
- Users can apply multiple tags (e.g., product review mentions camera AND battery AND performance)
- Documents belong to multiple domains

**Key parameter:** `cls_threshold` — lower = more labels assigned (higher recall); higher = fewer labels (higher precision).

In [8]:
# Sample: Product reviews — multiple aspects per review
reviews = [
    "The camera takes stunning 4K photos but the battery drains super fast. Performance is smooth though, no lag at all.",
    "Incredibly lightweight laptop with a gorgeous display. The keyboard feels mushy and the trackpad is a bit sluggish. Fan noise is minimal.",
    "Best headphones I've owned. Sound quality is exceptional, noise cancellation blocks out everything. Slightly uncomfortable after 2 hours of wear.",
    "This smartwatch tracks fitness accurately and the GPS is spot-on. The app could use improvement and the price feels steep for what it offers.",
]

print("=== Multi-label Review Aspect Classification ===")
print("(Multi-label picks ALL aspects mentioned, not just the dominant one)")
print()

for i, review in enumerate(reviews, 1):
    result = model.classify_text(
        review,
        {
            "aspects_mentioned": {
                "labels": ["camera", "battery", "performance", "display", "price",
                           "build_quality", "software", "sound", "comfort", "noise"],
                "multi_label": True,
                "cls_threshold": 0.35,
            },
            "overall_sentiment": ["positive", "negative", "mixed"],
            "recommend": ["yes", "no", "maybe"],
        }
    )
    print(f"  Review {i}: \"{review[:60]}...\"")
    print(f"    aspects   : {result['aspects_mentioned']}")
    print(f"    sentiment : {result['overall_sentiment']}")
    print(f"    recommend : {result['recommend']}")
    print()

=== Multi-label Review Aspect Classification ===
(Multi-label picks ALL aspects mentioned, not just the dominant one)

  Review 1: "The camera takes stunning 4K photos but the battery drains s..."
    aspects   : ['camera', 'battery', 'performance']
    sentiment : mixed
    recommend : yes

  Review 2: "Incredibly lightweight laptop with a gorgeous display. The k..."
    aspects   : ['performance', 'display', 'noise']
    sentiment : mixed
    recommend : yes

  Review 3: "Best headphones I've owned. Sound quality is exceptional, no..."
    aspects   : ['performance', 'sound', 'comfort', 'noise']
    sentiment : positive
    recommend : yes

  Review 4: "This smartwatch tracks fitness accurately and the GPS is spo..."
    aspects   : ['performance', 'price', 'software']
    sentiment : mixed
    recommend : maybe



In [9]:
# Comparing cls_threshold values on the same text
review = "The camera takes stunning photos, battery is decent, performance is excellent, display is bright, and the price is fair."

print("=== Effect of cls_threshold on Multi-label Output ===")
print(f'Text: "{review}"')
print()

for threshold in [0.2, 0.35, 0.5, 0.7]:
    result = model.classify_text(
        review,
        {
            "aspects": {
                "labels": ["camera", "battery", "performance", "display", "price", "comfort", "design"],
                "multi_label": True,
                "cls_threshold": threshold,
            }
        }
    )
    print(f"  threshold={threshold}: {result['aspects']}")

print()
print("Lower threshold = more labels (higher recall). Higher threshold = fewer labels (higher precision).")
print("Recommended starting point: 0.35–0.45 for multi-label.")

=== Effect of cls_threshold on Multi-label Output ===
Text: "The camera takes stunning photos, battery is decent, performance is excellent, display is bright, and the price is fair."

  threshold=0.2: ['camera', 'battery', 'performance', 'display', 'price']
  threshold=0.35: ['camera', 'battery', 'performance', 'display', 'price']
  threshold=0.5: ['camera', 'battery', 'performance', 'display', 'price']
  threshold=0.7: ['camera', 'battery', 'performance', 'display', 'price']

Lower threshold = more labels (higher recall). Higher threshold = fewer labels (higher precision).
Recommended starting point: 0.35–0.45 for multi-label.


---
## Section 6: Confidence Scores <a id='s06'></a>

GLiNER2 can return **how confident** it is about each extraction. Confidence is a float from 0 to 1.

**Why confidence scores matter:**
- Filter out low-confidence extractions before storing or acting on them
- Route borderline extractions to human review
- Monitor model quality over time
- Make smarter downstream decisions ("if domain_confidence < 0.7, don't apply domain filter")

**Parameter:** `include_confidence=True` on any extraction method.

In [10]:
# Classification confidence
news_headlines = [
    "Apple reports record Q4 earnings, stock jumps 12% after hours",
    "Mild weather expected across the midwest this weekend",
    "New study links coffee consumption to reduced Alzheimer's risk",
    "Team wins championship after dramatic overtime victory",   # ambiguous — sports or other?
]

print("=== Classification WITH Confidence Scores ===")
print(f"{'Headline':<55} {'Topic':<15} {'Confidence'}")
print("-" * 85)

for headline in news_headlines:
    result = model.classify_text(
        headline,
        {
            "topic": ["finance", "technology", "sports", "health", "weather", "politics", "science"]
        },
        include_confidence=True
    )
    topic_result = result["topic"]
    # With include_confidence, single-label result is {"label": ..., "confidence": ...}
    label = topic_result["label"]
    confidence = topic_result["confidence"]
    flag = "⚠️  LOW" if confidence < 0.6 else "✅"
    print(f"  {headline[:53]:<55} {label:<15} {confidence:.3f}  {flag}")

=== Classification WITH Confidence Scores ===
Headline                                                Topic           Confidence
-------------------------------------------------------------------------------------
  Apple reports record Q4 earnings, stock jumps 12% aft   finance         0.796  ✅
  Mild weather expected across the midwest this weekend   weather         1.000  ✅
  New study links coffee consumption to reduced Alzheim   health          0.934  ✅
  Team wins championship after dramatic overtime victor   sports          1.000  ✅


In [11]:
# Multi-label confidence
review = "Great camera, fast processor, terrible battery life, and overpriced for what you get."

result = model.classify_text(
    review,
    {
        "aspects": {
            "labels": ["camera", "performance", "battery", "price", "display", "design"],
            "multi_label": True,
            "cls_threshold": 0.3,
        }
    },
    include_confidence=True
)

print("=== Multi-label WITH Confidence ===")
print(f'Text: "{review}"')
print()
print(f"{'Aspect':<15} {'Confidence':>12}")
print("-" * 30)
# With include_confidence + multi_label, result is [{"label": ..., "confidence": ...}, ...]
for item in result["aspects"]:
    bar = "█" * int(item["confidence"] * 20)
    print(f"  {item['label']:<13} {item['confidence']:>8.3f}  {bar}")

=== Multi-label WITH Confidence ===
Text: "Great camera, fast processor, terrible battery life, and overpriced for what you get."

Aspect            Confidence
------------------------------
  camera           1.000  ███████████████████
  performance      0.753  ███████████████
  battery          1.000  ███████████████████
  price            1.000  ███████████████████


In [12]:
# Using confidence for smart filtering
texts_to_classify = [
    "The transformer architecture revolutionized natural language processing.",
    "Pass me the salt.",   # nonsense for tech classification — should have low confidence
    "Kubernetes simplifies container orchestration at scale.",
    "The 2024 Olympics were held in Paris.",
]

CONFIDENCE_THRESHOLD = 0.65

print(f"=== Confidence-based Filtering (threshold={CONFIDENCE_THRESHOLD}) ===")
print("Low-confidence results are flagged instead of blindly stored.")
print()

for text in texts_to_classify:
    result = model.classify_text(
        text,
        {"domain": ["machine_learning", "cloud_computing", "devops", "security", "database", "general"]},
        include_confidence=True
    )
    label = result["domain"]["label"]
    conf  = result["domain"]["confidence"]
    action = "STORE" if conf >= CONFIDENCE_THRESHOLD else "FLAG FOR REVIEW"
    print(f"  Text   : {text[:60]}")
    print(f"  Result : domain={label} (confidence={conf:.3f}) → {action}")
    print()

=== Confidence-based Filtering (threshold=0.65) ===
Low-confidence results are flagged instead of blindly stored.

  Text   : The transformer architecture revolutionized natural language
  Result : domain=machine_learning (confidence=1.000) → STORE

  Text   : Pass me the salt.
  Result : domain=general (confidence=1.000) → STORE

  Text   : Kubernetes simplifies container orchestration at scale.
  Result : domain=cloud_computing (confidence=0.997) → STORE

  Text   : The 2024 Olympics were held in Paris.
  Result : domain=general (confidence=1.000) → STORE



---
## Section 7: Structured Data Extraction — Basic <a id='s07'></a>

GLiNER2 can parse unstructured text into clean JSON with named fields — like having GPT-4 in JSON mode but running locally and for free.

**Method:** `model.extract_json(text, schema_dict)`

**Field specification syntax:**
```
"field_name::dtype::description"
```
- `dtype` = `str` (single value) or `list` (multiple values)
- `description` = plain English hint to improve accuracy

Let's pull structured data out of messy, real-world text.

In [13]:
# Sample: Unstructured job posting → structured JSON
job_posting = """
We're looking for a Senior ML Engineer to join our Core AI team at TechNova (Series B, $50M raised).
Location: San Francisco or Remote (US only). Salary: $180,000–$230,000 + equity.

You'll be working on large language models, fine-tuning pipelines, and inference optimization.
Must have: 5+ years experience, Python, PyTorch, and experience with transformer architectures.
Nice to have: Kubernetes, AWS SageMaker, Rust.

Perks: Unlimited PTO, $5,000 learning budget, top-of-the-line MacBook Pro, remote-first culture.
Apply by: March 31, 2025. Contact: careers@technova.ai
"""

result = model.extract_json(
    job_posting,
    {
        "job": [
            "title::str::Job title or role name",
            "company::str::Company name",
            "location::str::Office location or remote policy",
            "salary_range::str::Compensation range in USD",
            "required_skills::list::Must-have technical skills or years of experience",
            "nice_to_have_skills::list::Optional or preferred technical skills",
            "perks::list::Benefits, perks, or company culture mentions",
            "apply_deadline::str::Application deadline date",
            "contact::str::Email or contact information",
        ]
    }
)

print("=== Job Posting → Structured JSON ===")
pp(result)

=== Job Posting → Structured JSON ===
{
  "job": [
    {
      "title": "Senior ML Engineer",
      "company": "TechNova",
      "location": null,
      "salary_range": null,
      "required_skills": [
        "Python",
        "PyTorch",
        "experience with transformer architectures",
        "5+ years experience"
      ],
      "nice_to_have_skills": [
        "Kubernetes",
        "AWS SageMaker",
        "Rust"
      ],
      "perks": [
        "top-of-the-line MacBook Pro",
        "$5,000 learning budget",
        "remote-first culture",
        "Unlimited PTO"
      ],
      "apply_deadline": "March 31, 2025",
      "contact": "careers@technova.ai"
    }
  ]
}


In [14]:
# Sample: Real estate listing → structured
listing = """
Stunning 3-bedroom, 2-bathroom single-family home in Palo Alto, CA — just $2,450,000!
Built in 2018, this 1,850 sqft modern home features an open floor plan, chef's kitchen
with quartz countertops, hardwood floors throughout, and a 2-car garage.
The backyard has a heated pool and landscaped garden. Minutes from Stanford University
and top-rated schools. HOA: $350/month. Property tax: ~$24,500/year.
Listed by: Sarah Johnson, Century 21 Realty. Open house: Sunday Feb 9, 1–4 PM.
"""

result = model.extract_json(
    listing,
    {
        "property": [
            "price::str::Asking price of the property",
            "address::str::City and state location",
            "bedrooms::str::Number of bedrooms",
            "bathrooms::str::Number of bathrooms",
            "size_sqft::str::Square footage",
            "year_built::str::Year the property was built",
            "features::list::Key features, amenities, or notable characteristics",
            "hoa_fee::str::Monthly HOA fee if any",
            "listing_agent::str::Name and agency of the listing agent",
            "open_house::str::Open house date and time",
        ]
    }
)

print("=== Real Estate Listing → Structured JSON ===")
pp(result)

=== Real Estate Listing → Structured JSON ===
{
  "property": [
    {
      "price": "$2,450,000",
      "address": "Palo Alto, CA",
      "bedrooms": null,
      "bathrooms": "2",
      "size_sqft": "1,850 sqft",
      "year_built": "2018",
      "features": [
        "landscaped garden",
        "open floor plan",
        "heated pool",
        "quartz countertops",
        "hardwood floors throughout",
        "top-rated schools",
        "2-car garage",
        "chef's kitchen",
        "Stanford University"
      ],
      "hoa_fee": "$350/month",
      "listing_agent": "Sarah Johnson, Century 21 Realty",
      "open_house": "Sunday Feb 9, 1\u20134 PM"
    }
  ]
}


---
## Section 8: Structured Extraction with choices/enum <a id='s08'></a>

The `choices` parameter constrains a field to a **predefined set of values**, turning it into an inline classification within the structured output.

**Syntax in `extract_json`:**
```
"field_name::[option1|option2|option3]::dtype::description"
```

For `dtype="str"`: picks the single best match
For `dtype="list"`: picks all matches above threshold

This is powerful for fields that should have controlled vocabulary — severity levels, categories, status codes, etc.

In [15]:
# Sample: Restaurant reservation request
reservation_requests = [
    "Table for 6 at Nobu on Friday March 14th at 8pm. We'd prefer indoor seating. Two of us are vegan, one has a gluten allergy. Special occasion — anniversary.",
    "Need a table for 2 people this Saturday evening around 7:30. Outdoor patio if possible. No dietary restrictions. It's a first date so something romantic.",
    "Booking for a business lunch, 8 guests, next Wednesday at noon. Private dining room preferred. One vegetarian and one kosher in the group.",
]

print("=== Restaurant Reservations with choices ===")
for i, request in enumerate(reservation_requests, 1):
    result = model.extract_json(
        request,
        {
            "reservation": [
                "party_size::str::Number of guests",
                "date::str::Requested date",
                "time::str::Requested time",
                "seating::[indoor|outdoor|private_room|bar|no_preference]::str::Seating preference",
                "dietary::[vegan|vegetarian|gluten_free|kosher|halal|none]::list::Dietary restrictions",
                "occasion::[anniversary|birthday|business|first_date|family|casual|none]::str::Type of occasion",
            ]
        }
    )
    print(f"\nRequest {i}: \"{request[:70]}...\"")
    pp(result)

=== Restaurant Reservations with choices ===

Request 1: "Table for 6 at Nobu on Friday March 14th at 8pm. We'd prefer indoor se..."
{
  "reservation": [
    {
      "party_size": "6",
      "date": "Friday March 14th",
      "time": "8pm",
      "seating": "indoor",
      "dietary": [
        "vegan"
      ],
      "occasion": "anniversary"
    }
  ]
}

Request 2: "Need a table for 2 people this Saturday evening around 7:30. Outdoor p..."
{
  "reservation": [
    {
      "party_size": "2 people",
      "date": "this Saturday",
      "time": "7:30",
      "seating": "outdoor",
      "dietary": [
        "none"
      ],
      "occasion": "first_date"
    }
  ]
}

Request 3: "Booking for a business lunch, 8 guests, next Wednesday at noon. Privat..."
{
  "reservation": [
    {
      "party_size": "8",
      "date": "next Wednesday",
      "time": "noon",
      "seating": "private_room",
      "dietary": [
        "vegetarian"
      ],
      "occasion": "business"
    }
  ]
}


In [16]:
# Sample: Bug report → structured with controlled vocabulary
bug_reports = [
    "App crashes instantly when I try to upload files larger than 10MB. Happens 100% of the time on iOS 17. I've lost 2 hours of work because of this.",
    "The dark mode toggle in settings doesn't save between sessions. Minor annoyance but happens consistently on Chrome 120.",
    "I think the font on the dashboard could be slightly larger for accessibility. Not a bug, just a suggestion for the next release.",
]

print("=== Bug Reports with Controlled Vocabulary ===")
for i, report in enumerate(bug_reports, 1):
    result = model.extract_json(
        report,
        {
            "ticket": [
                "issue_summary::str::Brief description of the problem",
                "severity::[critical|high|medium|low|enhancement]::str::Issue severity level",
                "type::[crash|data_loss|ui_bug|performance|security|feature_request]::str::Type of issue",
                "reproducibility::[always|intermittent|once|unknown]::str::How often it happens",
                "platform::list::Affected platforms, OS, browsers, or devices",
                "impact::str::Business or user impact of the issue",
            ]
        }
    )
    print(f"\nReport {i}: \"{report[:65]}...\"")
    pp(result)

=== Bug Reports with Controlled Vocabulary ===

Report 1: "App crashes instantly when I try to upload files larger than 10MB..."
{
  "ticket": [
    {
      "issue_summary": null,
      "severity": "high",
      "type": "crash",
      "reproducibility": "intermittent",
      "platform": [
        "iOS 17"
      ],
      "impact": "lost 2 hours of work"
    }
  ]
}

Report 2: "The dark mode toggle in settings doesn't save between sessions. M..."
{
  "ticket": [
    {
      "issue_summary": null,
      "severity": "low",
      "type": "ui_bug",
      "reproducibility": "always",
      "platform": [
        "Chrome 120"
      ],
      "impact": null
    }
  ]
}

Report 3: "I think the font on the dashboard could be slightly larger for ac..."
{
  "ticket": [
    {
      "issue_summary": "font on the dashboard could be slightly larger",
      "severity": "enhancement",
      "type": "ui_bug",
      "reproducibility": "unknown",
      "platform": [
        "dashboard"
      ],
      "impact"

---
## Section 9: Per-field Confidence Thresholds <a id='s09'></a>

Different fields require different levels of precision:
- **High threshold (0.85+):** Account numbers, version strings, IBANs, dates — must be exact
- **Medium threshold (0.65–0.80):** Names, companies, locations — some tolerance
- **Low threshold (0.45–0.60):** Lists, features, descriptions — higher recall preferred

Using the **Schema Builder API** (instead of `extract_json`) gives you this fine-grained control.

In [17]:
# Sample: Financial transaction data — precision is critical!
transaction_text = """
Wire transfer confirmation: TXN-2025-887643
From: Apex Capital Management (Account: GB29NWBK60161331926819)
To: Vertex Trading LLC (Account: DE89370400440532013000)
Amount: EUR 2,450,000.00 transferred on 2025-01-15 at 14:23 UTC
Reference: INVOICE-2024-Q4-FINAL settlement
Exchange rate applied: 1 EUR = 1.0876 USD
Transaction fee: USD 125.00
Compliance note: Subject to OFAC screening, cleared by compliance officer J. Williams
"""

schema = (model.create_schema()
    .structure("transaction")
        .field("transaction_id",  dtype="str",
               description="Unique transaction ID or reference number",
               threshold=0.90)   # Must be exact — high precision
        .field("sender",          dtype="str",
               description="Sending party name",
               threshold=0.70)
        .field("sender_iban",     dtype="str",
               description="Sender IBAN or bank account number",
               threshold=0.90)   # Financial IDs must be high precision
        .field("receiver",        dtype="str",
               description="Receiving party name",
               threshold=0.70)
        .field("receiver_iban",   dtype="str",
               description="Receiver IBAN or bank account number",
               threshold=0.90)
        .field("amount",          dtype="str",
               description="Transfer amount with currency",
               threshold=0.85)
        .field("transfer_date",   dtype="str",
               description="Date and time of transfer",
               threshold=0.85)
        .field("fee",             dtype="str",
               description="Transaction fee charged",
               threshold=0.75)
        .field("compliance_flags",dtype="list",
               description="Compliance checks, regulatory notes, or risk flags",
               threshold=0.55)   # Lower — capture all compliance mentions
)

result = model.extract(transaction_text, schema)
print("=== Financial Transaction (per-field thresholds) ===")
pp(result)

=== Financial Transaction (per-field thresholds) ===
{
  "transaction": [
    {
      "transaction_id": "TXN-2025-887643",
      "sender": "Apex Capital Management",
      "sender_iban": "GB29NWBK60161331926819",
      "receiver": "Vertex Trading LLC",
      "receiver_iban": "DE89370400440532013000",
      "amount": "EUR 2,450,000.00",
      "transfer_date": "2025-01-15 at 14:23 UTC",
      "fee": "USD 125.00",
      "compliance_flags": [
        "OFAC screening"
      ]
    }
  ]
}


In [18]:
# Sample: Software release notes → version extraction needs high precision
release_notes = """
PostgreSQL 16.2 Released — February 8, 2024

This release fixes several security issues discovered in PostgreSQL 16.1 and PostgreSQL 15.5.
Users running versions prior to 16.1 are strongly encouraged to upgrade immediately.
PostgreSQL 14.x reaches end-of-life in November 2026.

Changes:
- CVE-2024-0985: Fixed privilege escalation vulnerability (affects 16.0, 16.1, 15.x, 14.x)
- Fixed crash in pg_dump with large schemas (reported in GitHub issue #7234)
- Improved performance of parallel query execution by approximately 15%
- Updated bundled OpenSSL to 3.2.1
"""

schema = (model.create_schema()
    .structure("release")
        .field("software",         dtype="str", threshold=0.80)
        .field("new_version",      dtype="str",
               description="The version being released in this announcement",
               threshold=0.88)
        .field("affected_versions",dtype="list",
               description="Older versions affected by these changes or vulnerabilities",
               threshold=0.82)
        .field("cves",             dtype="list",
               description="CVE security vulnerability identifiers",
               threshold=0.90)   # CVE IDs must be exact
        .field("improvements",     dtype="list",
               description="Performance improvements or feature enhancements",
               threshold=0.50)   # Lower — catch all improvement mentions
        .field("eol_dates",        dtype="list",
               description="End-of-life dates for older versions",
               threshold=0.80)
)

result = model.extract(release_notes, schema)
print("=== Software Release Notes (per-field thresholds) ===")
pp(result)

=== Software Release Notes (per-field thresholds) ===
{
  "release": [
    {
      "software": null,
      "new_version": null,
      "affected_versions": [
        "16.1",
        "14.x",
        "16.0",
        "15.x"
      ],
      "cves": [
        "CVE-2024-0985"
      ],
      "improvements": [
        "performance of parallel query execution"
      ],
      "eol_dates": [
        "November 2026"
      ]
    }
  ]
}


---
## Section 10: RegexValidator — Post-processing Filters <a id='s10'></a>

`RegexValidator` applies regex patterns **after** GLiNER2 extracts spans, filtering or excluding results that don't match expected formats.

**Three modes:**
| Mode | Behavior |
|------|---------|
| `"full"` | Entire extracted span must match the pattern |
| `"partial"` | Pattern must appear somewhere in the span |
| `"exclude"` | Spans MATCHING the pattern are REMOVED |

Regex validators are especially powerful for:
- Structured identifiers (emails, phone numbers, IBANs, IP addresses, version strings)
- Removing false positives (e.g., single-word nonsense extracted as a company name)

In [19]:
# Sample: Contact information extraction with validators
messy_contact_text = """
Please reach out to our team:
- Sales: john.doe@company.com or sarah.smith@company.co.uk
- Support: not-an-email or support AT company DOT com (not valid)
- Phone: +1 (415) 555-0123 or 415.555.0124
- Fax: 987-654-3210 (this format invalid for our system)
- Website: https://company.com and http://old.company.com
- Twitter: @companyhandle  (don't extract this as a URL)
"""

# Validators
email_validator = RegexValidator(
    pattern=r"^[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}$",
    mode="full"       # entire span must be a valid email
)
us_phone_validator = RegexValidator(
    pattern=r"\+?1?[\s\-\.]?\(?\d{3}\)?[\s\-\.]?\d{3}[\s\-\.]?\d{4}",
    mode="partial"    # pattern can appear within the span
)
https_validator = RegexValidator(
    pattern=r"^https?://",
    mode="partial"    # URL must start with http or https
)
no_social_validator = RegexValidator(
    pattern=r"^@",
    mode="partial",
    exclude=True      # exclude anything starting with @
)

schema = (model.create_schema()
    .structure("contacts")
        .field("emails",   dtype="list", validators=[email_validator])
        .field("phones",   dtype="list", validators=[us_phone_validator])
        .field("websites", dtype="list", validators=[https_validator, no_social_validator])
)

result = model.extract(messy_contact_text, schema)
print("=== Contact Extraction WITH Regex Validators ===")
pp(result)
print()
print("Validators filtered out: 'not-an-email', 'support AT company DOT com',")
print("'987-654-3210', '@companyhandle'")

=== Contact Extraction WITH Regex Validators ===
{
  "contacts": [
    {
      "emails": [
        "john.doe@company.com",
        "sarah.smith@company.co.uk"
      ],
      "phones": [
        "+1 (415) 555-0123",
        "415.555.0124"
      ],
      "websites": [
        "http://old.company.com",
        "https://company.com"
      ]
    }
  ]
}

Validators filtered out: 'not-an-email', 'support AT company DOT com',
'987-654-3210', '@companyhandle'


In [20]:
# RegexValidator to clean up version number extraction
# Without validator: "Chapter 2", "Section 1.1", "Figure 3" get extracted as versions
# With validator: only actual semver/version strings pass through

changelog_text = """
Django 5.1.3 (released January 2025)
Fixes a regression introduced in Django 5.1.2. Upgrade from Django 4.2.x is supported.
See Chapter 3 of the migration guide. Figure 1 shows the architecture.
Compatible with Python 3.10, 3.11, 3.12. Requires Pillow >= 9.4.0.
Breaking change from version 3.x: removed legacy authentication backend.
"""

# Without validator
result_raw = model.extract_json(
    changelog_text,
    {"release": ["versions_mentioned::list::Version numbers or version strings"]}
)
print("WITHOUT version validator:")
print(" ", result_raw.get("release", []))

# With regex validator for semver-like patterns
version_validator = RegexValidator(
    pattern=r"v?\d+(\.\d+){1,3}([-+][a-zA-Z0-9.]+)?",
    mode="partial"
)
# Also exclude standalone numbers that aren't versions
not_chapter_validator = RegexValidator(
    pattern=r"(?i)^(chapter|section|figure|table|page)\s+\d",
    mode="partial",
    exclude=True
)

schema = (model.create_schema()
    .structure("release")
        .field("versions_mentioned", dtype="list",
               description="Software version numbers or version strings",
               threshold=0.75,
               validators=[version_validator, not_chapter_validator])
        .field("python_requirements", dtype="list",
               description="Python version compatibility requirements",
               threshold=0.80)
)

result_clean = model.extract(changelog_text, schema)
print()
print("WITH version validator:")
pp(result_clean)

WITHOUT version validator:
  [{'versions_mentioned': ['Django 4.2.x', 'Django 5.1.3']}]

WITH version validator:
{
  "release": [
    {
      "versions_mentioned": [
        "Django 4.2.x",
        "Django 5.1.3",
        "Django 5.1.2"
      ],
      "python_requirements": [
        "Pillow >= 9.4.0"
      ]
    }
  ]
}


---
## Section 11: Multi-task Extraction — Everything in One Call <a id='s11'></a>

This is GLiNER2's most powerful feature: **entities + classification + structured data all extracted in a SINGLE model forward pass**.

**Benefits:**
- Text is encoded **once** — all tasks share that computation
- Faster than calling each task separately
- Coherent results — all tasks see the same context window simultaneously

**Method:** Build a schema with `create_schema()`, then call `model.extract(text, schema)`

In [21]:
# Sample: Earnings call transcript snippet
# We want: key people + companies (NER) + sentiment + topic (classification) + key metrics (structured)
earnings_transcript = """
CEO Lisa Chen opened Q3 2024 earnings call: "We delivered record revenue of $4.2 billion,
up 34% year-over-year. Our cloud segment grew 67% to $1.8 billion, driven by strong
enterprise adoption in EMEA and North America. Gross margins improved to 74.3% from 71.2%
last year. We're raising full-year guidance to $16.5–17.0 billion."

CFO Michael Torres added: "Operating expenses were well-controlled at $2.1 billion.
We ended the quarter with $8.4 billion in cash. We repurchased $500M in shares
and will continue our buyback program through 2025. Free cash flow was $1.2 billion."

Analyst David Kim from Goldman Sachs asked about competitive pressure from Microsoft Azure.
Chen responded: "We see continued share gains in the mid-market. Our win rate against Azure
improved 8 points quarter-over-quarter."
"""

# Multi-task schema: NER + 2 classifications + structured metrics — ONE call
schema = (model.create_schema()

    # Task 1: NER
    .entities({
        "executive":  "C-suite executives, senior leaders, or company officers mentioned",
        "company":    "Company names, divisions, or competitors mentioned",
        "analyst":    "Financial analysts, investors, or research firm representatives",
        "geography":  "Regions, countries, or markets mentioned (EMEA, North America, etc.)",
    })

    # Task 2: Overall sentiment classification
    .classification("earnings_sentiment",
        ["beat_expectations", "met_expectations", "missed_expectations", "mixed"])

    # Task 3: Key topic identification (multi-label)
    .classification("topics", {
        "labels": ["revenue_growth", "margin_expansion", "guidance_raise", "buyback",
                   "competitive_dynamics", "regional_performance", "cash_flow"],
        "multi_label": True,
        "cls_threshold": 0.35,
    })

    # Task 4: Structured financial metrics
    .structure("financials")
        .field("total_revenue",     dtype="str", description="Total revenue figure with growth rate", threshold=0.80)
        .field("cloud_revenue",     dtype="str", description="Cloud or SaaS segment revenue", threshold=0.80)
        .field("gross_margin",      dtype="str", description="Gross margin percentage", threshold=0.82)
        .field("annual_guidance",   dtype="str", description="Full-year revenue guidance range", threshold=0.80)
        .field("cash_position",     dtype="str", description="Cash and cash equivalents on balance sheet", threshold=0.80)
        .field("share_buyback",     dtype="str", description="Share repurchase amount", threshold=0.78)
        .field("key_metrics",       dtype="list", description="Other notable KPIs, percentages, or financial figures", threshold=0.55)
)

result = model.extract(earnings_transcript, schema)

print("=== Earnings Call: Multi-task in One Forward Pass ===")
print()
print("ENTITIES:")
pp(result.get("entities", {}))
print()
print("CLASSIFICATIONS:")
print(f"  earnings_sentiment : {result.get('earnings_sentiment')}")
print(f"  topics             : {result.get('topics')}")
print()
print("FINANCIALS:")
pp(result.get("financials", []))

=== Earnings Call: Multi-task in One Forward Pass ===

ENTITIES:
{
  "executive": [
    "Michael Torres",
    "Lisa Chen"
  ],
  "company": [
    "Microsoft Azure"
  ],
  "analyst": [
    "David Kim"
  ],
  "geography": [
    "EMEA",
    "North America"
  ]
}

CLASSIFICATIONS:
  earnings_sentiment : met_expectations
  topics             : labels

FINANCIALS:
[
  {
    "total_revenue": "$4.2 billion",
    "cloud_revenue": null,
    "gross_margin": "74.3%",
    "annual_guidance": null,
    "cash_position": "$8.4 billion",
    "share_buyback": "$500M",
    "key_metrics": [
      "Operating expenses",
      "Gross margins",
      "revenue",
      "win rate",
      "cloud segment",
      "Free cash flow"
    ]
  }
]


In [22]:
# Side-by-side: Multi-task in ONE call vs separate calls
sample_text = """Apple's Tim Cook unveiled Vision Pro 2 at WWDC 2025 in San Jose.
The $3,499 device features M4 chip and 8 hours battery. Analysts from Morgan Stanley
upgraded Apple to Buy with a $250 target."""

combined_schema = (model.create_schema()
    .entities({"person": "People", "company": "Companies", "product": "Products"})
    .classification("sentiment", ["bullish", "bearish", "neutral"])
    .structure("product")
        .field("name",  dtype="str")
        .field("price", dtype="str")
        .field("specs", dtype="list", description="Technical specifications")
)

print("=== Speed Comparison: One multi-task call vs three separate calls ===")

# Multi-task: 1 call
t0 = time.perf_counter()
result_combined = model.extract(sample_text, combined_schema)
t_combined = time.perf_counter() - t0

# Separate calls: 3 calls
t0 = time.perf_counter()
_ = model.extract_entities(sample_text, {"person": "People", "company": "Companies", "product": "Products"})
_ = model.classify_text(sample_text, {"sentiment": ["bullish", "bearish", "neutral"]})
_ = model.extract_json(sample_text, {"product": ["name::str", "price::str", "specs::list"]})
t_separate = time.perf_counter() - t0

print(f"  Multi-task (1 call)  : {t_combined*1000:.0f} ms")
print(f"  Separate (3 calls)   : {t_separate*1000:.0f} ms")
print(f"  Speedup              : {t_separate/t_combined:.1f}x faster")
print()
print("Multi-task result:")
pp(result_combined)

=== Speed Comparison: One multi-task call vs three separate calls ===
  Multi-task (1 call)  : 97 ms
  Separate (3 calls)   : 214 ms
  Speedup              : 2.2x faster

Multi-task result:
{
  "product": [
    {
      "name": "Vision Pro 2",
      "price": "$3,499",
      "specs": [
        "8 hours battery",
        "M4 chip"
      ]
    }
  ],
  "entities": {
    "person": [
      "Tim Cook"
    ],
    "company": [
      "Apple",
      "Morgan Stanley"
    ],
    "product": [
      "Vision Pro 2"
    ]
  },
  "sentiment": "bullish"
}


---
## Section 12: Batch Processing <a id='s12'></a>

For processing **many texts** efficiently, use batch methods instead of calling single-text methods in a loop.

**Why batch matters:**
The expensive part of GLiNER2 is the transformer encoder forward pass. Batch processing groups multiple texts into a single forward pass — dramatically reducing overhead.

```
# Slow: N encoder forward passes
for text in texts:
    result = model.extract(text, schema)

# Fast: ceil(N / batch_size) encoder forward passes
results = model.batch_extract(texts, schema, batch_size=8)
```

**Available batch methods:**
- `batch_extract(texts, schema, batch_size, threshold, include_confidence)`
- `batch_classify_text(texts, tasks, batch_size, threshold, include_confidence)`
- `batch_extract_json(texts, structures, batch_size, threshold, include_confidence)`

In [23]:
# Sample: E-commerce product catalogue (many products to process)
product_catalogue = [
    "Sony WH-1000XM5 Wireless Headphones — $349. Industry-leading noise cancellation, 30-hour battery, multipoint connection. Colors: black, silver.",
    "Kindle Paperwhite (16GB) — $139. 6.8-inch display, IPX8 waterproof, 3-month free Kindle Unlimited. Available in denim blue and agave green.",
    "Dyson V15 Detect Absolute Vacuum — $749. Laser reveals invisible dust, HEPA filtration captures 99.97% of particles, 60-min runtime on a single charge.",
    "Apple AirPods Pro (2nd Gen) — $249. Active Noise Cancellation, Transparency mode, Adaptive Audio, H2 chip. Up to 30 hours total listening time.",
    "Instant Pot Duo 7-in-1, 6 Quart — $89. Pressure cooker, slow cooker, rice cooker, steamer, sauté, yogurt maker, warmer. 13 one-touch programs.",
    "Nikon Z6 III Mirrorless Camera — $2,499. 24.5MP partial stacked CMOS sensor, 6K RAW video, 20fps burst, dual card slots. Body only.",
    "Logitech MX Master 3S Mouse — $99. 8K DPI Darkfield sensor, quiet clicks, fast USB-C charging, 70-day battery, works on glass surfaces.",
    "Vitamix 5200 Blender — $549. Variable speed control, self-cleaning in 30-60 seconds, 64 oz low-profile container, aircraft-grade stainless blades.",
]

print("=== Batch Processing: E-commerce Catalogue ===")
print(f"Processing {len(product_catalogue)} products with batch_size=4...")

t0 = time.perf_counter()
results = model.batch_extract_json(
    product_catalogue,
    {
        "product": [
            "name::str::Product name and model number",
            "price::str::Price in USD",
            "key_features::list::Top 3 most important product features",
            "colors::list::Available color options if mentioned",
            "battery_life::str::Battery life or runtime if applicable",
        ]
    },
    batch_size=4,
    threshold=0.5,
)
elapsed = time.perf_counter() - t0

print(f"Processed {len(results)} products in {elapsed:.1f}s ({elapsed/len(results)*1000:.0f} ms/product avg)")
print()
print("Results:")
for i, (product_text, result) in enumerate(zip(product_catalogue, results), 1):
    items = result.get("product", [{}])
    item = items[0] if items else {}
    print(f"  {i}. {item.get('name', '?'):<40} ${item.get('price', '?')}")
    print(f"     Features: {item.get('key_features', [])}")

=== Batch Processing: E-commerce Catalogue ===
Processing 8 products with batch_size=4...
Processed 8 products in 0.4s (52 ms/product avg)

Results:
  1. Sony WH-1000XM5 Wireless Headphones      $$349
     Features: ['multipoint connection', 'noise cancellation']
  2. Kindle Paperwhite                        $$139
     Features: ['IPX8 waterproof', '3-month free Kindle Unlimited']
  3. Dyson V15 Detect Absolute Vacuum         $$749
     Features: ['HEPA filtration', 'Laser']
  4. Apple AirPods Pro (2nd Gen)              $$249
     Features: ['Adaptive Audio', 'Transparency mode', 'Active Noise Cancellation', 'H2 chip']
  5. Instant Pot Duo 7-in-1                   $$89
     Features: ['one-touch programs']
  6. Nikon Z6 III                             $$2,499
     Features: ['20fps burst', '6K RAW video', 'dual card slots', '24.5MP partial stacked CMOS sensor']
  7. Logitech MX Master 3S Mouse              $None
     Features: ['quiet clicks', 'fast USB-C charging', 'Darkfield sensor',

In [24]:
# Batch classification — process many customer emails at once
customer_emails = [
    "I've been a customer for 5 years and this is absolutely unacceptable. My order #45123 was supposed to arrive last week and still nothing. I want a refund immediately.",
    "Just received my package and everything looks perfect! The packaging was beautiful and the product quality exceeded my expectations. Will definitely order again.",
    "Quick question — do you offer bulk discounts for orders over 100 units? We're a small business and this could work well for us.",
    "The website keeps crashing on Safari whenever I try to add items to cart. Using macOS Sonoma 14.2. This has been happening for 3 days.",
    "Can I change my delivery address? My order was placed yesterday, order #67890. I made a typo in the zip code.",
]

print("=== Batch Classification: Customer Emails ===")

t0 = time.perf_counter()
classifications = model.batch_classify_text(
    customer_emails,
    {
        "intent":    ["complaint", "praise", "inquiry", "bug_report", "order_change"],
        "urgency":   ["immediate", "high", "medium", "low"],
        "sentiment": ["angry", "happy", "neutral", "frustrated"],
    },
    batch_size=4,
)
elapsed = time.perf_counter() - t0
print(f"Classified {len(customer_emails)} emails in {elapsed*1000:.0f} ms total\n")

print(f"{'#':<3} {'Intent':<15} {'Urgency':<12} {'Sentiment':<12} {'Email preview'}")
print("-" * 85)
for i, (email, cls) in enumerate(zip(customer_emails, classifications), 1):
    print(f"  {i:<2} {cls['intent']:<15} {cls['urgency']:<12} {cls['sentiment']:<12} {email[:40]}...")

=== Batch Classification: Customer Emails ===
Classified 5 emails in 222 ms total

#   Intent          Urgency      Sentiment    Email preview
-------------------------------------------------------------------------------------
  1  complaint       immediate    angry        I've been a customer for 5 years and thi...
  2  praise          low          happy        Just received my package and everything ...
  3  inquiry         low          neutral      Quick question — do you offer bulk disco...
  4  complaint       high         frustrated   The website keeps crashing on Safari whe...
  5  order_change    high         neutral      Can I change my delivery address? My ord...


In [25]:
# Batch with confidence scores + timing comparison (loop vs batch)
texts = [
    "The Matrix was directed by the Wachowskis and stars Keanu Reeves as Neo.",
    "Beethoven composed the Ninth Symphony despite being completely deaf.",
    "Tesla was founded in 2003 by Martin Eberhard and Marc Tarpenning.",
    "Mount Everest, at 8,849 meters, was first summited by Hillary and Norgay in 1953.",
]

entity_types = ["person", "organization", "location", "work_of_art", "date", "event"]

# Method 1: Loop (slow)
t0 = time.perf_counter()
loop_results = [model.extract_entities(t, entity_types) for t in texts]
t_loop = time.perf_counter() - t0

# Method 2: Batch (fast)
t0 = time.perf_counter()
batch_results = model.batch_extract(
    texts,
    model.create_schema().entities(entity_types),
    batch_size=4,
    include_confidence=True,
)
t_batch = time.perf_counter() - t0

print(f"=== Loop vs Batch Speed Comparison ===")
print(f"  {len(texts)} texts | Loop: {t_loop*1000:.0f}ms | Batch: {t_batch*1000:.0f}ms | Speedup: {t_loop/t_batch:.1f}x")
print()
print("Batch result for text 1 (with confidence):")
pp(batch_results[0])

=== Loop vs Batch Speed Comparison ===
  4 texts | Loop: 265ms | Batch: 127ms | Speedup: 2.1x

Batch result for text 1 (with confidence):
{
  "entities": {
    "person": [
      {
        "text": "Keanu Reeves",
        "confidence": 0.987141489982605
      },
      {
        "text": "Wachowskis",
        "confidence": 0.6772691011428833
      }
    ],
    "organization": [],
    "location": [],
    "work_of_art": [
      {
        "text": "The Matrix",
        "confidence": 0.957216739654541
      }
    ],
    "date": [],
    "event": []
  }
}


---
## Section 13: Schema Builder API — Full Control <a id='s13'></a>

The **Schema Builder** is the full-power API. It gives you everything the convenience methods don't:
- Per-field thresholds
- RegexValidators
- Multi-label classification configuration
- Combining all tasks in one schema object
- **Reusable** — build once, use many times

The pattern: `model.create_schema().<chain of tasks>.field(...)`

In [26]:
# Sample: Full HR hiring pipeline — combine everything
cv_text = """
Alexandra Torres — Senior Data Scientist
Email: alex.torres@email.com | LinkedIn: linkedin.com/in/alextorres | Phone: +1 (628) 555-0199
San Francisco, CA

EXPERIENCE
Stripe Inc. (2021–present) — Lead Data Scientist
  - Built fraud detection model reducing chargebacks by 34%, saving $12M annually
  - Led team of 6 data scientists; managed $2M ML infrastructure budget
  - Stack: Python, PyTorch, Apache Spark, Kubernetes, AWS SageMaker, dbt

Airbnb (2018–2021) — Data Scientist II
  - Developed dynamic pricing algorithm increasing host revenue by 18%
  - Published 2 internal research papers on causal inference

EDUCATION
Stanford University — M.S. Computer Science (2018), GPA 3.9/4.0
UC Berkeley — B.S. Statistics (2016)

SKILLS: Python, R, SQL, PyTorch, TensorFlow, Spark, Kubernetes, AWS, GCP, dbt, Airflow
LANGUAGES: English (native), Spanish (fluent), Mandarin (basic)
"""

# Full schema with all features combined
hire_schema = (model.create_schema()

    # NER: extract people and organizations
    .entities({
        "person":       "Candidate full name",
        "company":      "Current or past employer names",
        "institution":  "Universities, schools, or educational institutions",
        "technology":   "Programming languages, frameworks, tools, and platforms",
    })

    # Seniority level classification
    .classification("seniority",
        ["intern", "junior", "mid_level", "senior", "lead", "principal", "director"])

    # Specialization (multi-label — can be both ML and MLOps)
    .classification("specializations", {
        "labels": ["machine_learning", "data_engineering", "mlops", "nlp",
                   "computer_vision", "statistics", "analytics", "research"],
        "multi_label": True,
        "cls_threshold": 0.35,
    })

    # Structured contact info with regex validators
    .structure("contact")
        .field("name",     dtype="str",  description="Full name of the candidate",         threshold=0.80)
        .field("email",    dtype="str",  description="Email address",                      threshold=0.85,
               validators=[RegexValidator(r"^[\w._%+\-]+@[\w.\-]+\.[a-zA-Z]{2,}$", mode="full")])
        .field("phone",    dtype="str",  description="Phone number",                       threshold=0.80)
        .field("location", dtype="str",  description="City and state",                     threshold=0.70)

    # Structured experience
    .structure("experience")
        .field("employer",         dtype="str",  description="Most recent company name",   threshold=0.78)
        .field("title",            dtype="str",  description="Current or most recent job title", threshold=0.75)
        .field("years_experience", dtype="str",  description="Total years of work experience",  threshold=0.65)
        .field("key_achievements", dtype="list", description="Quantified achievements with metrics", threshold=0.55)
        .field("team_size",        dtype="str",  description="Size of team managed if any",threshold=0.70)

    # Structured education
    .structure("education")
        .field("degree",       dtype="str",  description="Highest degree earned",          threshold=0.78)
        .field("field",        dtype="str",  description="Field of study or major",        threshold=0.70)
        .field("institution",  dtype="str",  description="University or school name",      threshold=0.78)
        .field("gpa",          dtype="str",  description="GPA or academic distinction",    threshold=0.82)
)

result = model.extract(cv_text, hire_schema)

print("=== Full HR CV Parsing (All Features Combined) ===")
print()
print("ENTITIES:")
pp(result.get("entities", {}))
print()
print("CLASSIFICATIONS:")
print(f"  seniority       : {result.get('seniority')}")
print(f"  specializations : {result.get('specializations')}")
print()
print("CONTACT:")
pp(result.get("contact", []))
print()
print("EXPERIENCE:")
pp(result.get("experience", []))
print()
print("EDUCATION:")
pp(result.get("education", []))

=== Full HR CV Parsing (All Features Combined) ===

ENTITIES:
{
  "person": [
    "Alexandra Torres"
  ],
  "company": [
    "Stripe Inc.",
    "Airbnb"
  ],
  "institution": [
    "UC Berkeley",
    "Stanford University"
  ],
  "technology": [
    "PyTorch",
    "Kubernetes",
    "Python",
    "dbt",
    "TensorFlow",
    "R",
    "Apache Spark",
    "AWS SageMaker",
    "SQL",
    "GCP",
    "AWS",
    "Spark",
    "Airflow"
  ]
}

CLASSIFICATIONS:
  seniority       : senior
  specializations : multi_label

CONTACT:
[
  {
    "name": "Alexandra Torres",
    "email": "alex.torres@email.com",
    "phone": "+1 (628) 555-0199",
    "location": "San Francisco, CA"
  }
]

EXPERIENCE:
[
  {
    "employer": "Stripe Inc.",
    "title": "Lead Data Scientist",
    "years_experience": "2021\u2013present",
    "key_achievements": [
      "saving $12M annually"
    ],
    "team_size": null
  },
  {
    "employer": "Airbnb",
    "title": "Data Scientist II",
    "years_experience": "2018\u20132021"

---
## Section 14: Schema Caching & Reuse <a id='s14'></a>

Building a schema is cheap, but the **best practice** is to build it **once** and reuse it across many texts. GLiNER2 caches the schema internally, so repeated use of the same `Schema` object avoids redundant processing.

This is especially important in production pipelines where you process thousands of documents with the same schema.

In [27]:
# Bad practice: build schema inside the loop (rebuilds every iteration)
docs = [
    "Amazon AWS launched a new EC2 instance type optimized for machine learning workloads.",
    "Google Cloud announced Gemini 2.0 Pro integration with Vertex AI in January 2025.",
    "Microsoft Azure expanded its data center presence in Southeast Asia with three new regions.",
]

print("=== Schema Caching: Bad vs Good Practice ===")

# Bad: rebuilds schema object every iteration
t0 = time.perf_counter()
for doc in docs:
    schema_per_call = (model.create_schema()  # rebuilds each time!
        .entities({"company": "Company names", "product": "Product or service names"})
        .classification("cloud_provider", ["aws", "gcp", "azure", "other"])
    )
    _ = model.extract(doc, schema_per_call)
t_bad = time.perf_counter() - t0

# Good: build once, reuse
shared_schema = (model.create_schema()   # built ONCE
    .entities({"company": "Company names", "product": "Product or service names"})
    .classification("cloud_provider", ["aws", "gcp", "azure", "other"])
)

t0 = time.perf_counter()
for doc in docs:
    _ = model.extract(doc, shared_schema)  # reuses cached schema
t_good = time.perf_counter() - t0

print(f"  Rebuild schema each call : {t_bad*1000:.0f} ms total")
print(f"  Reuse cached schema      : {t_good*1000:.0f} ms total")
print(f"  Benefit at scale         : ~{t_bad/t_good:.1f}x faster")
print()
print("In production: define schemas at module level, outside request handlers.")
print("Never build schemas inside a loop or per-request function.")

=== Schema Caching: Bad vs Good Practice ===
  Rebuild schema each call : 243 ms total
  Reuse cached schema      : 227 ms total
  Benefit at scale         : ~1.1x faster

In production: define schemas at module level, outside request handlers.
Never build schemas inside a loop or per-request function.


In [28]:
# Production pattern: Pre-built schemas as module-level constants
# Imagine these are defined at the top of your production module

# Schema 1: For processing tech articles
TECH_ARTICLE_SCHEMA = (model.create_schema()
    .entities({"technology": "Tools and frameworks", "company": "Companies", "person": "People"})
    .classification("domain", ["machine_learning", "cloud_computing", "devops", "security", "other"])
    .classification("content_type", ["tutorial", "news", "research", "opinion", "product_launch"])
    .structure("article_metadata")
        .field("key_technologies", dtype="list", threshold=0.60)
        .field("target_audience",  dtype="str",
               choices=["beginner", "intermediate", "advanced", "general"],
               threshold=0.65)
)

# Schema 2: For processing customer feedback
FEEDBACK_SCHEMA = (model.create_schema()
    .classification("sentiment",  ["positive", "negative", "neutral"])
    .classification("category",   ["product", "service", "billing", "shipping", "other"])
    .classification("action_required", ["refund", "escalate", "respond", "monitor", "none"])
)

# Use both schemas on appropriate texts
tech_text = "PyTorch 2.5 introduces new compilation backends that speed up LLM training by 40% on NVIDIA H100 GPUs."
feedback_text = "The package arrived damaged and customer service never responded to my emails. Extremely disappointed."

print("=== Pre-built Schemas in Action ===")
print()
print(f"Tech article: \"{tech_text[:60]}...\"")
pp(model.extract(tech_text, TECH_ARTICLE_SCHEMA))
print()
print(f"Customer feedback: \"{feedback_text[:60]}...\"")
pp(model.extract(feedback_text, FEEDBACK_SCHEMA))

=== Pre-built Schemas in Action ===

Tech article: "PyTorch 2.5 introduces new compilation backends that speed u..."
{
  "article_metadata": [
    {
      "key_technologies": [
        "PyTorch"
      ],
      "target_audience": "advanced"
    }
  ],
  "entities": {
    "technology": [
      "PyTorch"
    ],
    "company": [
      "NVIDIA"
    ],
    "person": []
  },
  "domain": "machine_learning",
  "content_type": "product_launch"
}

Customer feedback: "The package arrived damaged and customer service never respo..."
{
  "sentiment": "negative",
  "category": "service",
  "action_required": "respond"
}


---
## Section 15: base vs large — Accuracy vs Speed <a id='s15'></a>

| | `gliner2-base-v1` | `gliner2-large-v1` |
|-|-------------------|-------------------|
| **Parameters** | 205M | 340M |
| **Speed** | Faster (~1.5–2x) | Slower |
| **Accuracy** | Good | Better (especially on complex/ambiguous text) |
| **Memory** | ~800 MB RAM | ~1.3 GB RAM |
| **Best for** | High-throughput pipelines, prototyping | Production, complex schemas, ambiguous text |

In [29]:
# Load large model for comparison
# (Uncomment if you want to run — downloads ~500MB on first run)

# print("Loading GLiNER2 large model...")
# t0 = time.perf_counter()
# model_large = GLiNER2.from_pretrained("fastino/gliner2-large-v1")
# print(f"Large model loaded in {time.perf_counter() - t0:.1f}s")

# Comparison text — deliberately ambiguous to stress-test models
ambiguous_text = """
Apple's new M3 chip outperforms Intel's latest Core i9 by 40% in single-core benchmarks
according to Primate Labs (makers of Geekbench). Python developer Guido van Rossum
mentioned this at PyCon while discussing CPython's performance improvements in 3.12.
Meanwhile, Amazon's AWS Lambda now supports Apple Silicon natively.
"""

print("=== base model on ambiguous text ===")
t0 = time.perf_counter()
result_base = model.extract_entities(
    ambiguous_text,
    {
        "chip":       "CPU, GPU, or hardware chip products (e.g., M3, Core i9)",
        "company":    "Technology companies or organizations",
        "person":     "Individual people mentioned by name",
        "software":   "Programming languages, frameworks, or software products",
        "benchmark":  "Performance benchmarks or testing tools",
        "event":      "Conferences, events, or product launches",
    }
)
t_base = time.perf_counter() - t0
print(f"base model ({t_base*1000:.0f}ms):")
pp(result_base)

# Note: For large model comparison, uncomment and run with model_large
# t0 = time.perf_counter()
# result_large = model_large.extract_entities(ambiguous_text, {...same schema...})
# t_large = time.perf_counter() - t0
# print(f"\nlarge model ({t_large*1000:.0f}ms):")
# pp(result_large)

print()
print("When to use large model:")
print("  - 'Apple' must be company, not fruit")
print("  - 'Python' must be language, not snake")
print("  - Fine-grained entity boundary decisions")
print("  - High-stakes production (medical, legal, financial)")

=== base model on ambiguous text ===
base model (107ms):
{
  "entities": {
    "chip": [
      "Core i9",
      "M3"
    ],
    "company": [
      "Apple",
      "Intel",
      "Amazon"
    ],
    "person": [
      "Guido van Rossum"
    ],
    "software": [
      "CPython",
      "AWS Lambda",
      "Python"
    ],
    "benchmark": [
      "Geekbench"
    ],
    "event": [
      "PyCon"
    ]
  }
}

When to use large model:
  - 'Apple' must be company, not fruit
  - 'Python' must be language, not snake
  - Fine-grained entity boundary decisions
  - High-stakes production (medical, legal, financial)


---
## Section 16: Real-world Pipeline — News Intelligence Dashboard <a id='s16'></a>

Let's bring everything together. We'll build a mini **news intelligence pipeline** that:
1. Takes raw news articles as input
2. Classifies topic, sentiment, urgency (multi-label)
3. Extracts all key entities (NER)
4. Structures key data points (companies, metrics, dates)
5. Processes all articles in a single batch call
6. Outputs a clean intelligence report

This is what a production system looks like using GLiNER2.

In [30]:
# Five real news article snippets across different topics
news_articles = [
    """
    OpenAI raised $40 billion at a $300 billion valuation led by SoftBank in January 2025,
    marking the largest private tech fundraise in history. CEO Sam Altman confirmed the funds
    will be used to build larger AI infrastructure and expand data center capacity. Microsoft,
    an existing investor, contributed an additional $3 billion to the round.
    """,
    """
    The FDA approved Eli Lilly's oral GLP-1 weight loss pill, orforglipron, on February 3, 2025,
    the first once-daily pill in the class. Clinical trials showed 14.7% average body weight loss
    over 36 weeks. This could compete directly with Novo Nordisk's injectable Ozempic and Wegovy,
    which generated $14B in combined revenue in 2024. Lilly's stock rose 12% on the news.
    """,
    """
    A critical zero-day vulnerability (CVE-2025-0847) was disclosed in Cisco IOS XE affecting
    all versions prior to 17.12.3. The CVSS score is 9.8 (Critical). Over 50,000 devices
    are potentially exposed, according to Shodan research. CISA has added it to the Known
    Exploited Vulnerabilities catalog and mandated federal agencies to patch within 72 hours.
    """,
    """
    SpaceX's Starship completed its seventh test flight on January 16, 2025, successfully
    catching the Super Heavy booster with the mechazilla arms at Boca Chica for the second time.
    Elon Musk called it "a giant leap toward full and rapid reusability." NASA has contracted
    SpaceX for Artemis lunar lander missions in 2026 and 2027. The ship reached an altitude
    of 212 km before splashing down in the Indian Ocean.
    """,
    """
    Germany officially entered recession in Q4 2024, with GDP contracting 0.3% following
    a 0.2% decline in Q3. The Bundesbank blames high energy costs, weak Chinese demand for
    German exports, and structural challenges in the automotive sector. Volkswagen announced
    14,000 job cuts and BMW posted its first annual profit decline in a decade.
    Inflation remained sticky at 2.8%, complicating ECB rate cut decisions.
    """
]

# Build the intelligence pipeline schema
INTEL_SCHEMA = (model.create_schema()

    .entities({
        "organization": "Companies, government agencies, NGOs, or institutions",
        "person":       "Named individuals — executives, officials, researchers",
        "product":      "Products, drugs, technologies, vehicles, or systems",
        "location":     "Countries, cities, regions, or facilities",
        "regulation":   "Laws, standards, compliance mandates, CVE IDs, or regulatory bodies",
    })

    .classification("topic", [
        "ai_technology", "biotech_pharma", "cybersecurity",
        "space_aerospace", "economics_macro", "energy", "geopolitics"
    ])

    .classification("market_impact", [
        "highly_bullish", "bullish", "neutral", "bearish", "highly_bearish"
    ])

    .classification("urgency_tags", {
        "labels": ["breaking", "security_alert", "regulatory", "earnings_impact",
                   "earnings_impact", "m_and_a", "scientific_breakthrough"],
        "multi_label": True,
        "cls_threshold": 0.3,
    })

    .structure("key_facts")
        .field("headline_metric", dtype="str",
               description="The single most important number or statistic in this article",
               threshold=0.70)
        .field("key_companies",   dtype="list",
               description="The most important company names driving this story",
               threshold=0.65)
        .field("date_reference",  dtype="str",
               description="Primary date or time reference in the article",
               threshold=0.78)
        .field("financial_figures", dtype="list",
               description="Dollar amounts, valuations, revenue figures, or stock movements",
               threshold=0.72)
)

print("Running news intelligence pipeline on 5 articles...")
t0 = time.perf_counter()
intel_results = model.batch_extract(news_articles, INTEL_SCHEMA, batch_size=5)
elapsed = time.perf_counter() - t0
print(f"Processed in {elapsed:.1f}s ({elapsed/len(news_articles)*1000:.0f} ms/article)")

Running news intelligence pipeline on 5 articles...
Processed in 0.7s (134 ms/article)


In [31]:
# Format and display the intelligence report
article_titles = [
    "OpenAI $40B Fundraise",
    "FDA Approves Eli Lilly Oral GLP-1",
    "Cisco IOS XE Zero-day CVE-2025-0847",
    "SpaceX Starship Test Flight 7",
    "Germany Enters Recession Q4 2024",
]

print("=" * 80)
print("                   NEWS INTELLIGENCE REPORT")
print("=" * 80)

for i, (title, result) in enumerate(zip(article_titles, intel_results), 1):
    facts = result.get("key_facts", [{}])
    fact  = facts[0] if facts else {}
    entities = result.get("entities", {})

    print(f"\n[{i}] {title}")
    print(f"    Topic          : {result.get('topic', 'N/A')}")
    print(f"    Market Impact  : {result.get('market_impact', 'N/A')}")
    print(f"    Urgency Tags   : {result.get('urgency_tags', [])}")
    print(f"    Key Metric     : {fact.get('headline_metric', 'N/A')}")
    print(f"    Key Companies  : {fact.get('key_companies', [])}")
    print(f"    Date Ref       : {fact.get('date_reference', 'N/A')}")
    print(f"    Financial $    : {fact.get('financial_figures', [])}")
    print(f"    Orgs (NER)     : {entities.get('organization', [])}")
    print(f"    People (NER)   : {entities.get('person', [])}")
    if entities.get("regulation"):
        print(f"    Regulations    : {entities.get('regulation', [])}")

print()
print("=" * 80)
print("Pipeline complete. All 5 articles processed with:")
print("  - Named entity recognition (5 entity types)")
print("  - 3 classification tasks (topic, market impact, urgency tags)")
print("  - Structured fact extraction (4 fields per article)")
print("  - Everything in ONE batch call")
print("=" * 80)

                   NEWS INTELLIGENCE REPORT

[1] OpenAI $40B Fundraise
    Topic          : ai_technology
    Market Impact  : bullish
    Urgency Tags   : multi_label
    Key Metric     : largest private tech fundraise in history
    Key Companies  : ['OpenAI', 'Microsoft', 'SoftBank']
    Date Ref       : January 2025
    Financial $    : ['$40 billion', '$3 billion', '$300 billion']
    Orgs (NER)     : ['OpenAI', 'Microsoft', 'SoftBank']
    People (NER)   : ['Sam Altman']

[2] FDA Approves Eli Lilly Oral GLP-1
    Topic          : biotech_pharma
    Market Impact  : bullish
    Urgency Tags   : labels
    Key Metric     : 14.7% average body weight loss
    Key Companies  : ['Novo Nordisk', 'Eli Lilly', 'Wegovy', 'Ozempic']
    Date Ref       : None
    Financial $    : []
    Orgs (NER)     : ['Novo Nordisk', 'Eli Lilly', 'FDA']
    People (NER)   : []
    Regulations    : ['FDA']

[3] Cisco IOS XE Zero-day CVE-2025-0847
    Topic          : cybersecurity
    Market Impact  : bear

---
## Summary: Complete GLiNER2 Feature Map

| Feature | Method | Key Parameters | Best For |
|---------|--------|---------------|----------|
| **Basic NER** | `extract_entities(text, [types])` | `threshold`, `include_confidence` | Any domain, no training needed |
| **NER with descriptions** | `extract_entities(text, {type: desc})` | — | Ambiguous or technical domains |
| **Single-label classification** | `classify_text(text, {task: [labels]})` | `threshold`, `include_confidence` | Routing, tagging, filtering |
| **Multi-label classification** | `classify_text(text, {task: {labels, multi_label: True, cls_threshold}})` | `cls_threshold` | Reviews, multi-topic content |
| **Structured extraction** | `extract_json(text, {struct: [fields]})` | `threshold`, `include_confidence` | Forms, listings, profiles |
| **Enum/choice fields** | `"field::[a|b|c]::str"` syntax | — | Controlled vocabulary fields |
| **Per-field thresholds** | Schema Builder `.field(threshold=0.85)` | — | High-precision critical fields |
| **RegexValidator** | `.field(validators=[RegexValidator(...)])` | `mode`, `exclude`, `flags` | Format validation, noise removal |
| **Multi-task (all in one)** | `create_schema()` + `extract()` | — | Maximum efficiency |
| **Batch processing** | `batch_extract(texts, schema, batch_size)` | `batch_size`, `include_confidence` | High-throughput pipelines |
| **Confidence scores** | `include_confidence=True` on any method | — | Quality filtering, monitoring |
| **Schema caching** | Build `Schema` once, reuse | — | Production performance |

---

### The Key Mental Model

```
One model. One forward pass. Zero training.

You describe what you want → GLiNER2 extracts it.

vs. Traditional approach:
  - NER model (fine-tuned on your domain)
  + Classification model (fine-tuned per label set)
  + LLM API call (GPT-4 JSON mode, expensive)
  = 3 models, 3 calls, 3 bills

GLiNER2:
  = 1 model, 1 call, runs on your laptop CPU
```

### Resources
- [GitHub: fastino-ai/GLiNER2](https://github.com/fastino-ai/GLiNER2)
- [HuggingFace: fastino/gliner2-large-v1](https://huggingface.co/fastino/gliner2-large-v1)
- [arXiv paper (EMNLP 2025)](https://arxiv.org/abs/2507.18546)
- [PyPI: gliner2](https://pypi.org/project/gliner2/)